<a href="https://colab.research.google.com/github/ZackyRioOrlando/Praktikum-Pembelajaran-Mesin-/blob/main/JS09/JS09_Tugas_Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🎾
Tugas Lab 1

Lakukan percobaan pada metric distance yang berbeda, 1000 vs 1.000.000 data, 2D vs 5D data, untuk algoritma,

- ANNOY

- FAISS

- HNSW

Catat performansinya dalam bentuk tabel, misa

In [1]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 8.2 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 22.3 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520036 sha256=b4a397ce388f142bc9fde4baafd4387efdfd1cdcb221673d082c1d7a348e39bb
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679797 sha256=5a906484250da3247636120ed3fc794da81fc08dead0ba41db2dce1007827b81
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [3]:

import numpy as np
import pandas as pd
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ==========================================
# PARAMETER PENGUJIAN
# ==========================================
k = 10
jumlah_query = 20

skenario = [
    (1000, 2, "euclidean"),
    (1000, 2, "cosine"),
    (1000000, 5, "euclidean"),
    (1000000, 5, "cosine")
]

hasil = []

# ==========================================
# FUNGSI BENCHMARK
# ==========================================
def pengujian(n_data, dim, metric):

    print("\n===================================")
    print(f"Data: {n_data} | Dimensi: {dim}D")
    print(f"Metric: {metric}")
    print("===================================")

    # Membuat dataset
    X = np.random.random((n_data, dim)).astype(np.float32)
    Q = np.random.random((jumlah_query, dim)).astype(np.float32)

    # Normalisasi untuk cosine
    if metric == "cosine":
        X /= np.linalg.norm(X, axis=1, keepdims=True)
        Q /= np.linalg.norm(Q, axis=1, keepdims=True)

    # ======================================
    # 1. ANNOY
    # ======================================
    print("\n--- ANNOY ---")

    if metric == "euclidean":
        ann = AnnoyIndex(dim, "euclidean")
    else:
        ann = AnnoyIndex(dim, "angular")

    start = time.time()

    for i in range(n_data):
        ann.add_item(i, X[i])

    ann.build(10)

    build_ann = time.time() - start

    start = time.time()

    for q in Q:
        ann.get_nns_by_vector(q, k)

    query_ann = (time.time() - start) / jumlah_query

    # ======================================
    # 2. FAISS
    # ======================================
    print("--- FAISS ---")

    if metric == "euclidean":
        index = faiss.IndexFlatL2(dim)
    else:
        index = faiss.IndexFlatIP(dim)

    start = time.time()

    index.add(X)

    build_faiss = time.time() - start

    start = time.time()

    distances, indices = index.search(Q, k)

    query_faiss = (time.time() - start) / jumlah_query

    # ======================================
    # 3. HNSW
    # ======================================
    print("--- HNSW ---")

    if metric == "euclidean":
        hnsw_metric = "l2"
    else:
        hnsw_metric = "cosine"

    hnsw = hnswlib.Index(space=hnsw_metric, dim=dim)

    start = time.time()

    hnsw.init_index(
        max_elements=n_data,
        ef_construction=200,
        M=16
    )

    hnsw.add_items(X)

    build_hnsw = time.time() - start

    hnsw.set_ef(50)

    start = time.time()

    labels, distances = hnsw.knn_query(Q, k=k)

    query_hnsw = (time.time() - start) / jumlah_query

    # ======================================
    # MENAMPILKAN HASIL
    # ======================================
    print("\nHasil pengujian:")
    print("Annoy Build :", round(build_ann, 4), "detik")
    print("Annoy Query :", round(query_ann * 1000, 4), "ms")

    print("FAISS Build :", round(build_faiss, 4), "detik")
    print("FAISS Query :", round(query_faiss * 1000, 4), "ms")

    print("HNSW Build  :", round(build_hnsw, 4), "detik")
    print("HNSW Query  :", round(query_hnsw * 1000, 4), "ms")

    # Menyimpan hasil
    hasil.append({
        "Data/Dimensi": f"{n_data}/{dim}D",
        "Metric": metric,
        "ANNOY Build (s)": round(build_ann, 4),
        "ANNOY Query (ms)": round(query_ann * 1000, 4),
        "FAISS Build (s)": round(build_faiss, 4),
        "FAISS Query (ms)": round(query_faiss * 1000, 4),
        "HNSW Build (s)": round(build_hnsw, 4),
        "HNSW Query (ms)": round(query_hnsw * 1000, 4)
    })


# ==========================================
# MENJALANKAN SELURUH SKENARIO
# ==========================================
for n_data, dim, metric in skenario:
    pengujian(n_data, dim, metric)

# ==========================================
# TABEL AKHIR
# ==========================================
df = pd.DataFrame(hasil)

print("\n========== TABEL HASIL AKHIR ==========")
display(df)



Data: 1000 | Dimensi: 2D
Metric: euclidean

--- ANNOY ---
--- FAISS ---
--- HNSW ---

Hasil pengujian:
Annoy Build : 0.0647 detik
Annoy Query : 0.0406 ms
FAISS Build : 0.0001 detik
FAISS Query : 0.3996 ms
HNSW Build  : 0.1026 detik
HNSW Query  : 0.2096 ms

Data: 1000 | Dimensi: 2D
Metric: cosine

--- ANNOY ---
--- FAISS ---
--- HNSW ---

Hasil pengujian:
Annoy Build : 0.1216 detik
Annoy Query : 0.033 ms
FAISS Build : 0.0001 detik
FAISS Query : 0.1578 ms
HNSW Build  : 0.1346 detik
HNSW Query  : 0.1161 ms

Data: 1000000 | Dimensi: 5D
Metric: euclidean

--- ANNOY ---
--- FAISS ---
--- HNSW ---

Hasil pengujian:
Annoy Build : 19.5629 detik
Annoy Query : 0.08 ms
FAISS Build : 0.015 detik
FAISS Query : 5.093 ms
HNSW Build  : 158.8877 detik
HNSW Query  : 0.0455 ms

Data: 1000000 | Dimensi: 5D
Metric: cosine

--- ANNOY ---
--- FAISS ---
--- HNSW ---

Hasil pengujian:
Annoy Build : 27.5078 detik
Annoy Query : 0.0624 ms
FAISS Build : 0.0068 detik
FAISS Query : 6.4282 ms
HNSW Build  : 148.7104 d

,Data/Dimensi,Metric,ANNOY Build (s),ANNOY Query (ms),FAISS Build (s),FAISS Query (ms),HNSW Build (s),HNSW Query (ms)
0,1000/2D,euclidean,0.0647,0.0406,0.0001,0.3996,0.1026,0.2096
1,1000/2D,cosine,0.1216,0.0330,0.0001,0.1578,0.1346,0.1161
2,1000000/5D,euclidean,19.5629,0.0800,0.0150,5.0930,158.8877,0.0455
3,1000000/5D,cosine,27.5078,0.0624,0.0068,6.4282,148.7104,0.0500
